In [1]:
!cd .. && pip install . --group=dev --group=observability

Processing /home/jovyan/work
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for massi: filename=massi-0.1.0-py3-none-any.whl size=42326 sha256=0b66421570ea4290262c6a0f54f612160b47a8336bb422efdf8a2fc6739ddab6
  Stored in directory: /tmp/pip-ephem-wheel-cache-7jx66mw8/wheels/b3/ef/92/093562baef6fce9b92cfc59927bf9e82e306ed148e873389c0
Successfully built massi
  Attempting uninstall: massi
    Found existing installation: massi 0.1.0
    Uninstalling massi-0.1.0:
      Successfully uninstalled massi-0.1.0


In [2]:
!pip install grandalf

In [3]:
# Setup observability
import os
from lmnr import Laminar

Laminar.initialize(
    project_api_key=os.environ["LMNR_PROJECT_API_KEY"],
    base_url=f"http://{os.environ["HOST_MACHINE_URL_CORE"]}",
    http_port=int(os.environ["LMNR_PROJECT_HTTP_PORT"]),
    grpc_port=int(os.environ["LMNR_PROJECT_GRPC_PORT"]),
)

In [4]:
from langchain_ollama import ChatOllama

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, Sequence
import operator

In [5]:
from massi.agents.interfaces import Config, Dependencies
from massi.agents.massi import create_ollama_llm

config = Config(**{"model_url": "http://host.docker.internal:11434", "model_name": "granite4:3b"})
deps = Dependencies(llm_client=create_ollama_llm(config))

In [6]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[HumanMessage], operator.add]

llm = deps.llm_client

def call_model(state: AgentState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {"messages": [response]} # Append new message

# Define a new graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.set_entry_point("agent")
workflow.add_edge("agent", END)

graph = workflow.compile()

In [7]:
import grandalf

graph.get_graph().print_ascii()

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +-------+    
  | agent |    
  +-------+    
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [8]:
inputs = {"messages": [HumanMessage(content="Hi there!")]}
result = graph.invoke(inputs)

In [9]:
print(result['messages'][-1].content)

Hello! How can I assist you today?
